# 06 Backward and Gradients

This notebook audits backward capture: `backward_ready`, gradient saving, `GradFn` and `GradFnCall` records, and backward validation. A `GradFn` is TorchLens metadata for one PyTorch autograd node; a `GradFnCall` is one recorded backward hook/call event.

The model returns a scalar loss so a real `.backward()` call is simple and deterministic.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import inspect

import torch
from torch import nn
import torchlens as tl
from torchlens.validation import validate_backward_pass

torch.manual_seed(29)


class BackwardNet(nn.Module):
    """Small scalar-output model for backward capture."""

    def __init__(self) -> None:
        """Initialize layers."""

        super().__init__()
        self.stem = nn.Linear(3, 4)
        self.head = nn.Linear(4, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the model and return a scalar loss.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Scalar loss-like output.
        """

        hidden = torch.relu(self.stem(x))
        return self.head(hidden).pow(2).mean()


model = BackwardNet().eval()
x = torch.randn(5, 3, requires_grad=True)
print(f"tl.trace has backward_ready: {'backward_ready' in inspect.signature(tl.trace).parameters}")
print(f"Trace.log_backward signature: {inspect.signature(tl.Trace.log_backward)}")

`backward_ready=True` keeps the graph connected for a later backward pass. `gradients_to_save="all"` asks TorchLens to retain gradient payloads where it can.

In [ ]:
trace = tl.trace(
    model,
    x,
    backward_ready=True,
    gradients_to_save="all",
)
loss_layer = trace[trace.output_layers[0]]
print(f"loss layer: {loss_layer.layer_label}")
print(f"loss value: {loss_layer.out.item():.6f}")
print(f"loss requires_grad: {loss_layer.out.requires_grad}")
print(f"saved grad ops before backward: {len(trace.saved_grad_ops)}")

Calling `Trace.log_backward(loss)` runs PyTorch backward and records TorchLens backward metadata.

In [ ]:
trace.log_backward(loss_layer.out)
print(f"input .grad shape: {tuple(x.grad.shape)}")
print(f"input .grad norm: {x.grad.norm().item():.6f}")
print(f"grad fns: {len(trace.grad_fns)}")
print(f"grad fn calls: {len(trace.grad_fn_calls)}")
print(f"saved grad ops: {len(trace.saved_grad_ops)}")

Forward `Op` records can now expose saved gradients. This cell prints the first few gradient-bearing operations.

In [ ]:
for label in list(trace.saved_grad_ops.keys())[:5]:
    op = trace[label]
    grad = op.grad
    grad_shape = tuple(grad.shape) if isinstance(grad, torch.Tensor) else None
    print(f"{label:16s} func={op.func_name:8s} grad_shape={grad_shape} saved={grad is not None}")

`GradFn` records describe autograd nodes. `GradFnCall` records describe individual backward call events, including whether a gradient payload was saved.

In [ ]:
for grad_fn in list(trace.grad_fns)[:4]:
    print(
        f"GradFn {grad_fn.label:18s} class={grad_fn.class_name:18s} "
        f"op_label={grad_fn.op_label} calls={list(grad_fn.calls)}"
    )

print("--- calls ---")
for grad_call in list(trace.grad_fn_calls)[:4]:
    print(
        f"GradFnCall {grad_call.call_label:18s} "
        f"parent={grad_call.label:18s} saved={grad_call.is_saved}"
    )

`validate_backward_pass` compares TorchLens backward capture with ordinary PyTorch autograd. We import it from `torchlens.validation` to avoid the deprecated top-level shim.

In [ ]:
validation_model = BackwardNet().eval()
validation_model.load_state_dict(model.state_dict())
validation_x = x.detach().clone().requires_grad_(True)
valid = validate_backward_pass(
    validation_model,
    validation_x,
    loss_fn=lambda output: output,
    random_seed=29,
    validate_metadata=False,
)
print(f"validate_backward_pass: {valid}")

> NOTE: The glossary names `GradFn.grad_fn_class_name` and `Op.has_saved_gradient`, but this build exposes the public class name on `GradFn.class_name` and uses `op.grad is not None` as the executable saved-gradient check. Forward `Op` records still expose `grad_fn_class_name`.

## 🔧 Sandbox

Change `sandbox_batch` or the loss expression and inspect how many gradient records are saved.

In [ ]:
# TODO: try sandbox_batch = 2 or change the loss to output.abs().mean().
sandbox_batch = 3
sandbox_x = torch.randn(sandbox_batch, 3, requires_grad=True)
sandbox_trace = tl.trace(model, sandbox_x, backward_ready=True, gradients_to_save="all")
sandbox_loss = sandbox_trace[sandbox_trace.output_layers[0]].out
sandbox_trace.log_backward(sandbox_loss)
print(f"sandbox loss: {sandbox_loss.item():.6f}")
print(f"sandbox input grad norm: {sandbox_x.grad.norm().item():.6f}")
print(f"sandbox saved grad ops: {len(sandbox_trace.saved_grad_ops)}")
print(f"sandbox grad fn calls: {len(sandbox_trace.grad_fn_calls)}")